# Person identification

In [ ]:
import numpy as np
import torch

from models.DopplerDataset import * 
from models.Architectures import *
from utilities.training import *
from utilities.plotting import *
from utilities.data_processing import *

%load_ext autoreload
%autoreload 2

IDEA:
- Training with datasets S1 (P1) and S3 (P2) (Training-test split)
- Test with S4 and S5 

May be valid? The problem is that from training to test, there is a change environment/monitor position. Be aware of input covariance shift

In [ ]:
# Creating dataset for person identification task
DEBUG_MODE                 = True

TRAINING_TEST_DATASET_PATH  = "doppler_traces/S[1,3]*"
TEST_DATASET_PATH   = "doppler_traces/S[2,5]*"
DS_NAME = "person_identification"
DOPPLER_TRACE_SIZE  = 340

person_key = {'S1':1, 'S2':1, 'S3':2, 'S4':1, 'S5':2, 'S6':1, 'S7':3} # Maps dataset to corresponding person

LABELS     = ['W', 'E', 'R', 'J', 'L']
ACTIVITIES = ['Walking', 'Empty', 'Running', 'Jumping', 'Sitting']
PERSON_LABELS = [1, 2]
person_labels_map = {
    person: index
    for index, person in enumerate(PERSON_LABELS)
}

create_train_test_split(
    dataset_path=TRAINING_TEST_DATASET_PATH,
    doppler_trace_size=DOPPLER_TRACE_SIZE,
    activity_list=LABELS,
    train_ratio=0.7,
    ds_name=DS_NAME,
    seed=0,
)

############################## CHANGE train_test_split because it's shuffling test samples and we want to 
############################## keep the order for the evaluation of the model
##############################

In [ ]:
def person_label_from_filename(filename):
    source_set = filename[:2]  # Returns the first two characters of the filename, which indicate the source set (e.g., 'S1', 'S2', etc.)
    return person_key[source_set]

batch_size = 128
augmentation = "hv"
train_dataset = DopplerDataset(
    f"doppler_traces_{DS_NAME}_train",
    person_labels_map,
    db_conversion=True,
    normalization=False,
    augmentation=augmentation,
    label_function=person_label_from_filename,
)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = DopplerDataset(
    f"doppler_traces_{DS_NAME}_test",
    person_labels_map,
    db_conversion=True,
    normalization=False,
    label_function=person_label_from_filename,
)
test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=False)

train_dataset.getInfo()
test_dataset.getInfo()

In [ ]:
# Define SHARP model (changed)
batchNorm = False
sharp_model_noBN = SHARP(n_features=len(PERSON_LABELS), batchNorm=batchNorm)
device = \
    torch.device("mps") if torch.backends.mps.is_available() else \
    torch.device("cuda") if torch.cuda.is_available() else \
    torch.device("cpu")
learning_rate = 1e-4
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=sharp_model_noBN.parameters(), lr=learning_rate)

if DEBUG_MODE:
    #### CHECKING DATA TYPES 
    x, y = next(iter(train_dataloader))
    print(x.dtype)
    print(next(sharp_model_noBN.parameters()).dtype)
    ### DEVICE
    print(f"Device: {device}")

In [ ]:
# Training of the model
epochs = 50
train_loss, test_loss, train_acc, test_acc, _ = train_model(sharp_model_noBN, train_dataloader, test_dataloader, epochs, loss_fn, optimizer, device, verbosity=False)

In [ ]:
# Results (metrics and plot cm)